In [6]:
import ee
import geemap
import ipywidgets as widgets
from ipyleaflet import WidgetControl
import pandas as pd
import math
import traitlets
import logging

# --- PROTECCIÓN CONTRA TRAITERROR / ERRORES DE SINCRO DE WIDGETS ---
logging.getLogger('traitlets').setLevel(logging.ERROR)

# 1. Inicialización de Google Earth Engine
try:
    ee.Initialize(project='booming-list-511608-s2')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='booming-list-511608-s2')

# 2. Inicializar el mapa limpio (SIN capas extrañas)
Map = geemap.Map(center=[22.58, -102.52], zoom=13, height='680px')
Map.layers = []

# Cargar la colección de Sentinel-2 Harmonized filtrada
sentinel_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterDate('2026-01-01', '2026-10-01')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)))

imagen_base = sentinel_collection.median()

# Añadir OpenStreetMap como base
Map.add_basemap('OpenStreetMap')

# 3. Widgets de control flotantes con tamaños fijos para evitar el colapso visual
index_selector = widgets.Dropdown(
    options=[
        ('NDVI (Índice de Vegetación Normalizado)', 'NDVI'),
        ('NDWI (Índice de Agua Normalizado)', 'NDWI'),
        ('MNDWI (Índice de Agua Modificado)', 'MNDWI'),
        ('EVI (Índice de Vegetación Mejorado)', 'EVI'),
        ('NDSI (Índice de Nieve Normalizado)', 'NDSI'),
        ('BSI (Índice de Suelo Desnudo)', 'BSI'),
        ('SAVI (Índice de Vegetación Ajustado al Suelo)', 'SAVI'),
        ('NDMI (Índice de Humedad Normalizado)', 'NDMI')
    ],
    value='MNDWI',
    description='Estudio:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

input_radio = widgets.FloatText(
    value=50.0,
    description='Distancia(m):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='130px')
)

tipo_forma_marker = widgets.Dropdown(
    options=[('Círculo', 'circulo'), ('Cuadrado', 'cuadrado')],
    value='cuadrado',
    description='Marcador:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='125px')
)

figura_selector = widgets.Dropdown(
    options=[('Sin figuras', None)],
    value=None,
    description='Zona Activa:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

btn_procesar = widgets.Button(
    description='Procesar Zona',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='120px')
)

btn_exportar = widgets.Button(
    description='Exportar Píxeles',
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='125px')
)

btn_limpiar = widgets.Button(
    description='Limpiar Todo',
    button_style='danger',
    icon='trash',
    layout=widgets.Layout(width='115px')
)

control_leyenda_actual = None
control_indicador_actual = None

# Estructura global para acumular todas las figuras procesadas y sus metadatos
historial_geometrias = []

# Fórmulas oficiales de índices centralizadas
def obtener_imagen_indice(estudio):
    if estudio == 'NDVI':
        img = imagen_base.normalizedDifference(['B8', 'B4']).rename('NDVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#8c510a', '#dfc27d', '#f5f5f5', '#80cdc1', '#01665e']}
        etiquetas = ['Muy Densa (0.8)', 'Densa (0.6)', 'Moderada (0.4)', 'Baja (0.2)', 'Suelo/Escaso (0.0)']
    elif estudio == 'NDWI':
        img = imagen_base.normalizedDifference(['B3', 'B8']).rename('NDWI')
        vis = {'min': -0.3, 'max': 0.5, 'palette': ['#d73027', '#fee08b', '#ffffbf', '#91bfdb', '#4575b4']}
        etiquetas = ['Agua Alta (0.5)', 'Agua Media (0.3)', 'Humedad (0.0)', 'Transicion (-0.1)', 'Suelo Seco (-0.3)']
    elif estudio == 'MNDWI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        vis = {'min': -0.5, 'max': 0.4, 'palette': ['#a50026', '#fdae61', '#ffffbf', '#abd9e9', '#08519c']}
        etiquetas = ['Agua Libre / Mar (0.4)', 'Agua Somera (0.2)', 'Transicion (0.0)', 'Suelo (-0.2)', 'Urbano / Roca (-0.5)']
    elif estudio == 'EVI':
        img = imagen_base.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': imagen_base.select('B8'), 'RED': imagen_base.select('B4'), 'BLUE': imagen_base.select('B2')}
        ).rename('EVI')
        vis = {'min': 0.0, 'max': 1.0, 'palette': ['#f7fcb9', '#addd8e', '#78c679', '#31a354', '#006837']}
        etiquetas = ['Muy Alta (1.0)', 'Alta (0.75)', 'Media (0.5)', 'Baja (0.25)', 'Minima (0.0)']
    elif estudio == 'NDSI':
        img = imagen_base.normalizedDifference(['B3', 'B11']).rename('NDSI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#000000', '#c6dbef', '#9ecae1', '#4292c6', '#ffffff']}
        etiquetas = ['Nieve Alta (0.8)', 'Media (0.6)', 'Baja (0.4)', 'Minima (0.2)', 'Sin Nieve (0.0)']
    elif estudio == 'BSI':
        img = imagen_base.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
            {'SWIR': imagen_base.select('B11'), 'RED': imagen_base.select('B4'), 'NIR': imagen_base.select('B8'), 'BLUE': imagen_base.select('B2')}
        ).rename('BSI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#3288bd', '#abdda4', '#ffffbf', '#fdae61', '#d53e4f']}
        etiquetas = ['Suelo Denso (0.4)', 'Alto (0.2)', 'Neutro (0.0)', 'Bajo (-0.2)', 'Vegetacion/Agua (-0.4)']
    elif estudio == 'SAVI':
        img = imagen_base.expression(
            '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
            {'NIR': imagen_base.select('B8'), 'RED': imagen_base.select('B4')}
        ).rename('SAVI')
        vis = {'min': 0.0, 'max': 0.8, 'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#225ea8', '#0c2c84']}
        etiquetas = ['Alta (0.8)', 'Media-Alta (0.6)', 'Media (0.4)', 'Baja (0.2)', 'Minima (0.0)']
    elif estudio == 'NDMI':
        img = imagen_base.normalizedDifference(['B8', 'B11']).rename('NDMI')
        vis = {'min': -0.4, 'max': 0.4, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#4575b4']}
        etiquetas = ['Alta Humedad (0.4)', 'Media-Alta (0.2)', 'Moderada (0.0)', 'Baja (-0.2)', 'Muy Seco (-0.4)']
    return img, vis, estudio, etiquetas

def actualizar_indicador_activo(nombre_zona):
    global control_indicador_actual
    if control_indicador_actual is not None:
        try:
            Map.remove_control(control_indicador_actual)
        except Exception:
            pass

    html_indicador = f"""
    <div style="background: rgba(255, 255, 255, 0.95); padding: 8px 15px; border-radius: 8px; box-shadow: 0 0 15px rgba(0,0,0,0.3); font-family: sans-serif; font-size: 12px; font-weight: bold; color: #007bff; text-align: center;">
        {nombre_zona}
    </div>
    """
    indicador_widget = widgets.HTML(value=html_indicador)
    control_indicador_actual = WidgetControl(widget=indicador_widget, position='topright')
    Map.add_control(control_indicador_actual)

# 4. Función de análisis espacial
def ejecutar_analisis_individual(b):
    global control_leyenda_actual, historial_geometrias
    estudio = index_selector.value
    distancia_m = input_radio.value
    forma_marker = tipo_forma_marker.value

    geometria_base = None
    nombre_figura = "Desconocido"

    try:
        if hasattr(Map, 'draw_features') and len(Map.draw_features) > 0:
            idx = len(Map.draw_features)
            ultimo_dibujo = Map.draw_features[-1]

            if hasattr(ultimo_dibujo, 'geometry'):
                g = ultimo_dibujo.geometry()
            elif isinstance(ultimo_dibujo, dict):
                g = ee.Geometry(ultimo_dibujo.get('geometry', ultimo_dibujo))
            else:
                g = ee.Feature(ultimo_dibujo).geometry()

            tipo_g = g.type().getInfo()

            if tipo_g == 'Point':
                coords = g.coordinates().getInfo()
                lon, lat = coords[0], coords[1]

                if forma_marker == 'cuadrado':
                    delta_lat = distancia_m / 111320.0
                    delta_lon = distancia_m / (111320.0 * math.cos(math.radians(lat)))
                    
                    min_lon, max_lon = lon - delta_lon, lon + delta_lon
                    min_lat, max_lat = lat - delta_lat, lat + delta_lat
                    
                    geometria_base = ee.Geometry.BBox(min_lon, min_lat, max_lon, max_lat)
                    nombre_figura = f"Cuadrado_Lado{int(distancia_m*2)}m_{idx}"
                else:
                    geometria_base = g.buffer(distancia_m)
                    nombre_figura = f"Circulo_Radio{int(distancia_m)}m_{idx}"
            else:
                geometria_base = g
                nombre_figura = f"Figura_{idx}_{tipo_g}"

    except Exception as e:
        print(f"Error al capturar la figura: {e}")

    if geometria_base is None:
        print("Atención: Dibuja primero una geometría o coloca un marcador antes de procesar.")
        return

    layer_img, vis, titulo_leyenda, etiquetas = obtener_imagen_indice(estudio)
    nombre_completo_zona = f"{nombre_figura}_{estudio}"

    historial_geometrias.append({
        'geometry': geometria_base,
        'nombre': nombre_completo_zona
    })

    figura_selector.options = [(item['nombre'], item['geometry']) for item in historial_geometrias]
    figura_selector.value = geometria_base

    imagen_recortada = layer_img.clip(geometria_base)
    Map.addLayer(imagen_recortada, vis, nombre_completo_zona)

    if control_leyenda_actual is not None:
        try:
            Map.remove_control(control_leyenda_actual)
        except Exception:
            pass

    html_leyenda = f"""
    <div style="background: white; padding: 10px; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2); font-family: sans-serif; font-size: 10px; width: 230px;">
        <b>{titulo_leyenda}</b><br>
        <div style="background: linear-gradient(to top, {vis['palette'][0]}, {vis['palette'][1]}, {vis['palette'][2]}, {vis['palette'][3]}, {vis['palette'][4]}); width: 18px; height: 120px; float: left; margin-right: 8px; margin-top: 5px;"></div>
        <div style="height: 120px; display: inline-block; line-height: 24px;">
            <span>{etiquetas[0]}<br></span>
            <span>{etiquetas[1]}<br></span>
            <span>{etiquetas[2]}<br></span>
            <span>{etiquetas[3]}<br></span>
            <span>{etiquetas[4]}</span>
        </div>
    </div>
    """
    legend_widget = widgets.HTML(value=html_leyenda)
    control_leyenda_actual = WidgetControl(widget=legend_widget, position='bottomleft')
    Map.add_control(control_leyenda_actual)

    actualizar_indicador_activo(nombre_completo_zona)

    if geometria_base is not None:
        try:
            Map.centerObject(geometria_base, zoom=14)
        except Exception:
            pass

    print(f'Proceso completado para {nombre_figura} con el índice: {titulo_leyenda}.')

btn_procesar.on_click(lambda b: ejecutar_analisis_individual(b))

def al_cambiar_figura(change):
    if change['type'] == 'change' and change['name'] == 'value':
        geom_seleccionada = change['new']
        if geom_seleccionada is not None:
            for item in historial_geometrias:
                if item['geometry'] == geom_seleccionada:
                    actualizar_indicador_activo(item['nombre'])
                    try:
                        Map.centerObject(geom_seleccionada, zoom=14)
                    except Exception:
                        pass
                    break

figura_selector.observe(al_cambiar_figura)

# 7. Panel de controles flotante (definido antes para poder excluirlo al limpiar)
panel_controles = widgets.VBox([
    widgets.HBox([index_selector, figura_selector]),
    widgets.HBox([input_radio, tipo_forma_marker, btn_procesar, btn_exportar, btn_limpiar])
], layout=widgets.Layout(min_width='630px', min_height='90px', padding='4px'))

control_menu = WidgetControl(widget=panel_controles, position='bottomright')
Map.add_control(control_menu)

# 5. Función de limpieza total reforzada (elimina capas, historial, selector, leyendas e indicador superior)
def limpiar_todo_sistema(b):
    global historial_geometrias, control_leyenda_actual, control_indicador_actual
    
    historial_geometrias = []
    figura_selector.options = [('Sin figuras', None)]
    figura_selector.value = None
    
    Map.layers = []
    Map.add_basemap('OpenStreetMap')
    
    if hasattr(Map, 'draw_features'):
        Map.draw_features = []

    if control_leyenda_actual is not None:
        try:
            Map.remove_control(control_leyenda_actual)
        except Exception:
            pass
        control_leyenda_actual = None

    if control_indicador_actual is not None:
        try:
            Map.remove_control(control_indicador_actual)
        except Exception:
            pass
        control_indicador_actual = None

    # Barrido extra de seguridad para cualquier WidgetControl rezagado en el mapa
    for control in list(Map.controls):
        if isinstance(control, WidgetControl) and control != control_menu:
            try:
                Map.remove_control(control)
            except Exception:
                pass

    print("🗑️ Sistema limpiado por completo: capas, historial, selector, leyenda y etiquetas superiores eliminados.")

btn_limpiar.on_click(limpiar_todo_sistema)

# 6. Función de exportación con formato horizontal e IDs independientes
def exportar_pixeles_zona(b):
    if len(historial_geometrias) == 0:
        print("Atención: Primero procesa al menos una zona en el mapa.")
        return

    estudio = index_selector.value
    layer_img, _, _, _ = obtener_imagen_indice(estudio)

    print(f"Extrayendo y auditando píxeles en formato horizontal con IDs independientes...")

    try:
        dfs_figuras = []
        reporte_auditoria = []
        
        for idx, item in enumerate(historial_geometrias):
            geom_actual = item['geometry']
            nombre_origen = item['nombre']

            conteo_real_ee = layer_img.select(estudio).clip(geom_actual).reduceRegion(
                reducer=ee.Reducer.count(),
                geometry=geom_actual,
                scale=10,
                maxPixels=1e9
            ).getInfo().get(estudio, 0)

            region_sample = layer_img.addBands(ee.Image.pixelCoordinates('EPSG:4326')).clip(geom_actual).sample(
                region=geom_actual,
                scale=10,
                geometries=True,
                numPixels=4999
            ).limit(4999)

            print(f"Procesando zona: {nombre_origen} (Píxeles reales en GEE: {conteo_real_ee})...")
            features = region_sample.getInfo()['features']
            pixeles_agregados = len(features)
            
            pixeles_faltantes = max(0, conteo_real_ee - pixeles_agregados)

            filas_figura = []
            for i, feat in enumerate(features):
                props = feat['properties']
                coords = feat['geometry']['coordinates']
                
                filas_figura.append({
                    f'Pixel_ID_{nombre_origen}': f'px{idx+1}_{i}',
                    f'Longitud_{nombre_origen}': coords[0],
                    f'Latitud_{nombre_origen}': coords[1],
                    f'{estudio}_{nombre_origen}': props.get(estudio, None)
                })
            
            if len(filas_figura) > 0:
                dfs_figuras.append(pd.DataFrame(filas_figura))

            reporte_auditoria.append({
                'Figura': nombre_origen,
                'Pixeles_Totales_Reales': conteo_real_ee,
                'Pixeles_Descargados_En_CSV': pixeles_agregados,
                'Pixeles_Faltantes_Por_Limite': pixeles_faltantes
            })

        if len(dfs_figuras) == 0:
            print("No se encontraron píxeles en las zonas procesadas.")
            return

        df_final = pd.concat(dfs_figuras, axis=1)
        csv_name = f'analisis_{estudio.lower()}_multizona_horizontal.csv'
        df_final.to_csv(csv_name, index=False)

        df_audit = pd.DataFrame(reporte_auditoria)
        audit_name = f'auditoria_limite_pixeles_{estudio.lower()}.csv'
        df_audit.to_csv(audit_name, index=False)

        print(df_audit.to_string(index=False))
        print(f"¡Exportación exitosa!")
        print(f" Archivo horizontal con IDs independientes: '{csv_name}'")
        print(f" Archivo de auditoría: '{audit_name}'")

    except Exception as ex:
        print(f"Error en la extracción: {ex}")

btn_exportar.on_click(exportar_pixeles_zona)

Map

Map(center=[22.58, -102.52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…